In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(42)

merchants = {
    "Groceries":     (["BigBasket", "Reliance Fresh", "More Supermarket", "Local Kirana"], 300, 2500),
    "Food":          (["Swiggy", "Zomato", "Saravana Bhavan", "Cafe Coffee Day"], 150, 900),
    "Transport":     (["Uber", "Ola", "Chennai Metro", "Indian Oil Petrol"], 80, 1500),
    "Bills":         (["TNEB Electricity", "Airtel Recharge", "ACT Broadband"], 299, 2500),
    "Shopping":      (["Amazon", "Flipkart", "Myntra"], 400, 4000),
    "Entertainment": (["BookMyShow", "Netflix", "Spotify"], 119, 800),
}
weights = [0.25, 0.30, 0.20, 0.07, 0.12, 0.06]

rows = []
start = pd.Timestamp("2026-04-01")
for _ in range(600):
    cat = rng.choice(list(merchants), p=weights)
    names, lo, hi = merchants[cat]
    day = start + pd.Timedelta(days=int(rng.integers(0, 180)))
    hour = int(np.clip(rng.normal(15, 3.5), 7, 23))
    rows.append({
        "datetime": day + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60))),
        "merchant": rng.choice(names),
        "category": cat,
        "amount": round(float(rng.uniform(lo, hi)), 2),
        "city": "Chennai" if rng.random() < 0.95 else "Dindigul",
        "channel": rng.choice(["UPI", "Card", "UPI", "UPI"]),
        "is_injected_fraud": 0,
    })

frauds = [
    ("2026-06-14 02:47", "QuickPay Wallet XJ", "Unknown", 48500, "Delhi", "Card"),
    ("2026-07-03 03:12", "Intl Electronics Ltd", "Shopping", 72999, "Singapore", "Card"),
    ("2026-07-03 03:15", "Intl Electronics Ltd", "Shopping", 68999, "Singapore", "Card"),
    ("2026-08-21 01:58", "Crypto Exchange 24", "Unknown", 35000, "Mumbai", "UPI"),
    ("2026-09-02 04:05", "Gift Card Store", "Shopping", 25000, "Kolkata", "Card"),
    ("2026-09-02 04:06", "Gift Card Store", "Shopping", 25000, "Kolkata", "Card"),
    ("2026-09-10 23:55", "Unknown UPI Merchant", "Unknown", 19999, "Noida", "UPI"),
    ("2026-09-25 02:20", "Betting App Pay", "Unknown", 15000, "Hyderabad", "UPI"),
]
for dt, m, c, a, city, ch in frauds:
    rows.append({"datetime": pd.Timestamp(dt), "merchant": m, "category": c,
                 "amount": a, "city": city, "channel": ch, "is_injected_fraud": 1})

statement = pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)
statement.to_csv("../data/sample_statement.csv", index=False)

print(statement.shape)
print("Injected frauds:", statement["is_injected_fraud"].sum())
statement.tail()

(608, 7)
Injected frauds: 8


,datetime,merchant,category,amount,city,channel,is_injected_fraud
603,2026-09-26 15:19:00,Ola,Transport,926.66,Chennai,UPI,0
604,2026-09-26 18:41:00,Cafe Coffee Day,Food,219.59,Chennai,Card,0
605,2026-09-26 19:01:00,Swiggy,Food,778.04,Chennai,UPI,0
606,2026-09-27 11:13:00,Local Kirana,Groceries,821.54,Chennai,UPI,0
607,2026-09-27 11:47:00,TNEB Electricity,Bills,1597.76,Chennai,Card,0


In [2]:
st = pd.read_csv("../data/sample_statement.csv", parse_dates=["datetime"])

st["hour"] = st["datetime"].dt.hour
st["is_night"] = st["hour"].between(0, 5).astype(int)
st["log_amount"] = np.log1p(st["amount"])

median_amt = st["amount"].median()
st["amount_vs_usual"] = st["amount"] / median_amt

merchant_counts = st["merchant"].map(st["merchant"].value_counts())
st["merchant_is_rare"] = (merchant_counts <= 2).astype(int)

city_counts = st["city"].map(st["city"].value_counts())
st["city_is_rare"] = (city_counts <= 2).astype(int)

st = st.sort_values("datetime")
st["mins_since_prev"] = st["datetime"].diff().dt.total_seconds().div(60).fillna(1e4)
st["rapid_repeat"] = (st["mins_since_prev"] < 10).astype(int)

features = ["log_amount", "hour", "is_night", "amount_vs_usual",
            "merchant_is_rare", "city_is_rare", "rapid_repeat"]
st[features].describe().round(2)

,log_amount,hour,is_night,amount_vs_usual,merchant_is_rare,city_is_rare,rapid_repeat
count,608.00,608.00,608.00,608.00,608.00,608.00,608.00
mean,6.77,14.26,0.01,1.90,0.01,0.01,0.05
std,0.88,3.64,0.11,6.00,0.11,0.11,0.22
min,4.54,1.00,0.00,0.11,0.00,0.00,0.00
25%,6.21,12.00,0.00,0.60,0.00,0.00,0.00
50%,6.72,14.00,0.00,1.00,0.00,0.00,0.00
75%,7.31,17.00,0.00,1.79,0.00,0.00,0.00
max,11.20,23.00,1.00,87.78,1.00,1.00,1.00


In [3]:
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso.fit(st[features])

raw = -iso.score_samples(st[features])
st["risk_score"] = ((raw - raw.min()) / (raw.max() - raw.min()) * 100).round()

cols = ["datetime", "merchant", "amount", "city", "risk_score", "is_injected_fraud"]
st.sort_values("risk_score", ascending=False)[cols].head(15)

,datetime,merchant,amount,city,risk_score,is_injected_fraud
337,2026-07-03 03:15:00,Intl Electronics Ltd,68999.00,Singapore,100.0,1
336,2026-07-03 03:12:00,Intl Electronics Ltd,72999.00,Singapore,97.0,1
536,2026-09-02 04:06:00,Gift Card Store,25000.00,Kolkata,95.0,1
264,2026-06-14 02:47:00,QuickPay Wallet XJ,48500.00,Delhi,95.0,1
496,2026-08-21 01:58:00,Crypto Exchange 24,35000.00,Mumbai,94.0,1
600,2026-09-25 02:20:00,Betting App Pay,15000.00,Hyderabad,91.0,1
535,2026-09-02 04:05:00,Gift Card Store,25000.00,Kolkata,91.0,1
556,2026-09-10 23:55:00,Unknown UPI Merchant,19999.00,Noida,87.0,1
532,2026-09-01 14:38:00,Flipkart,3146.45,Chennai,62.0,0
280,2026-06-18 07:46:00,BigBasket,2423.92,Dindigul,61.0,0


In [4]:
st["flagged"] = (st["risk_score"] >= 80).astype(int)

tn, fp, fn, tp = confusion_matrix(st["is_injected_fraud"], st["flagged"]).ravel()
print(f"Risk ≥ 80 → caught {tp}/{tp+fn} hidden frauds, false alarms {fp}")

st["level"] = pd.cut(st["risk_score"], bins=[-1, 59, 79, 100],
                     labels=["🟢 Normal", "🟡 Check", "🔴 High risk"])
st["level"].value_counts()

Risk ≥ 80 → caught 8/8 hidden frauds, false alarms 0


level
🟢 Normal       598
🔴 High risk      8
🟡 Check          2
Name: count, dtype: int64

## Findings
- Unsupervised Isolation Forest on 7 engineered, per-user features (no labels used in training).
- Synthetic 6-month statement: 600 genuine + 8 injected frauds.
- Risk ≥ 80 caught 8/8 frauds with 0 false alarms; frauds scored 87–100, the highest genuine transaction scored 62.
- 2 genuine transactions scored "Check" (60–79): larger-than-usual or early-morning payments.
- Limitation: injected frauds are deliberately obvious; subtle fraud (normal amount, usual time and place) would not be caught.